# Build the LIB Table

This notebook creates the document-level `LIB` table for the constitution corpus. Each row in `LIB` represents one constitution as a source document and stores metadata that can be used later for summarization, filtering, comparison across countries, and model interpretation. The table combines metadata derived from the raw text files with summary counts derived from the parsed corpus.


## Inputs and Outputs

**Inputs**
- `individual_constitutions/*.txt`: raw constitution files
- `corpus.csv`: parsed token-level corpus used to compute article, clause, and token counts
- `V-Dem-CY-Core-v16.csv`: local V-Dem country-year data keyed to the `country_id` field used to add regime variables for each document's amended year

**Output**
- `lib.csv`: one row per constitution

The final `LIB` columns are:
- `doc_id`, `country_id`
- `year_created`, `year_amended`
- `region`, `region_compressed`
- `source_file`, `file_format`
- `char_count`, `line_count`
- `article_count`, `clause_count`, `token_count`
- `v2x_regime`, `v2x_freexp_altinf`, `v2x_rule`
- `v2x_regime_cat`, `v2x_freexp_altinf_cat`, `v2x_rule_cat`


## Imports and Document Header Pattern

The notebook uses the raw text files to build document-level metadata. The header regex extracts the country name stored in `country_id`, the original constitutional year (`year_created`), and any later revision text from the first line. A filename pattern is also used to recover the amended/current version year (`year_amended`) from filenames such as `Jamaica_1994.txt`.


In [11]:
from pathlib import Path
import re

import pandas as pd

# Configure the small set of regex patterns used to recover document-level metadata.
# Match the first-line document header and fallback filename year patterns.
RE_DOC_HEADER = re.compile(r"^(.+?)\s+(\d{4})(?:\s+\((.*)\))?$")
RE_FILENAME_YEAR = re.compile(r"^(?P<country_id>.+)_(?P<year>\d{4})$")
RE_ANY_YEAR = re.compile(r"\d{4}")


In [12]:
# Map each country_id to a broad geographic region for document metadata.
REGION_MAP = {
    'Afghanistan': 'Southern Asia',
    'Albania': 'Southern Europe',
    'Algeria': 'Northern Africa',
    'Andorra': 'Southern Europe',
    'Angola': 'Middle Africa',
    'Antigua and Barbuda': 'Caribbean',
    'Argentina': 'South America',
    'Armenia': 'Western Asia',
    'Australia': 'Australia and New Zealand',
    'Austria': 'Western Europe',
    'Azerbaijan': 'Western Asia',
    'Bahamas': 'Caribbean',
    'Bahrain': 'Western Asia',
    'Bangladesh': 'Southern Asia',
    'Barbados': 'Caribbean',
    'Belarus': 'Eastern Europe',
    'Belgium': 'Western Europe',
    'Belize': 'Central America',
    'Benin': 'Western Africa',
    'Bhutan': 'Southern Asia',
    'Bolivia': 'South America',
    'Bosnia-Herzegovina': 'Southern Europe',
    'Botswana': 'Southern Africa',
    'Brazil': 'South America',
    'Brunei': 'South-eastern Asia',
    'Bulgaria': 'Eastern Europe',
    'Burkina Faso': 'Western Africa',
    'Burundi': 'Eastern Africa',
    'Cambodia': 'South-eastern Asia',
    'Cameroon': 'Middle Africa',
    'Canada': 'Northern America',
    'Cape Verde': 'Western Africa',
    'Central African Republic': 'Middle Africa',
    'Chad': 'Middle Africa',
    'Chile': 'South America',
    'China': 'Eastern Asia',
    'Colombia': 'South America',
    'Comoros': 'Eastern Africa',
    'Congo': 'Middle Africa',
    'Costa Rica': 'Central America',
    'Cote D\'Ivoire': 'Western Africa',
    'Croatia': 'Southern Europe',
    'Cuba': 'Caribbean',
    'Cyprus': 'Western Asia',
    'Czech Republic': 'Eastern Europe',
    'Democratic Republic of the Congo': 'Middle Africa',
    'Denmark': 'Northern Europe',
    'Djibouti': 'Eastern Africa',
    'Dominica': 'Caribbean',
    'Dominican Republic': 'Caribbean',
    'East Timor': 'South-eastern Asia',
    'Ecuador': 'South America',
    'Egypt': 'Northern Africa',
    'El Salvador': 'Central America',
    'Equatorial Guinea': 'Middle Africa',
    'Eritrea': 'Eastern Africa',
    'Estonia': 'Northern Europe',
    'Ethiopia': 'Eastern Africa',
    'Fiji': 'Melanesia',
    'Finland': 'Northern Europe',
    'France': 'Western Europe',
    'Gabon': 'Middle Africa',
    'Gambia': 'Western Africa',
    'Georgia': 'Western Asia',
    'German Federal Republic': 'Western Europe',
    'Ghana': 'Western Africa',
    'Greece': 'Southern Europe',
    'Grenada': 'Caribbean',
    'Guatemala': 'Central America',
    'Guinea': 'Western Africa',
    'Guinea-Bissau': 'Western Africa',
    'Guyana': 'South America',
    'Haiti': 'Caribbean',
    'Honduras': 'Central America',
    'Hungary': 'Eastern Europe',
    'Iceland': 'Northern Europe',
    'India': 'Southern Asia',
    'Indonesia': 'South-eastern Asia',
    'Iran': 'Southern Asia',
    'Iraq': 'Western Asia',
    'Ireland': 'Northern Europe',
    'Israel': 'Western Asia',
    'Italy': 'Southern Europe',
    'Jamaica': 'Caribbean',
    'Japan': 'Eastern Asia',
    'Jordan': 'Western Asia',
    'Kazakhstan': 'Central Asia',
    'Kenya': 'Eastern Africa',
    'Kiribati': 'Micronesia',
    'Kosovo': 'Southern Europe',
    'Kuwait': 'Western Asia',
    'Kyrgyz Republic': 'Central Asia',
    'Laos': 'South-eastern Asia',
    'Latvia': 'Northern Europe',
    'Lebanon': 'Western Asia',
    'Lesotho': 'Southern Africa',
    'Liberia': 'Western Africa',
    'Libya': 'Northern Africa',
    'Liechtenstein': 'Western Europe',
    'Lithuania': 'Northern Europe',
    'Luxembourg': 'Western Europe',
    'Macedonia': 'Southern Europe',
    'Madagascar': 'Eastern Africa',
    'Malawi': 'Eastern Africa',
    'Malaysia': 'South-eastern Asia',
    'Maldives': 'Southern Asia',
    'Mali': 'Western Africa',
    'Malta': 'Southern Europe',
    'Marshall Islands': 'Micronesia',
    'Mauritania': 'Western Africa',
    'Mauritius': 'Eastern Africa',
    'Mexico': 'Central America',
    'Micronesia': 'Micronesia',
    'Moldova': 'Eastern Europe',
    'Monaco': 'Western Europe',
    'Mongolia': 'Eastern Asia',
    'Montenegro': 'Southern Europe',
    'Morocco': 'Northern Africa',
    'Mozambique': 'Eastern Africa',
    'Myanmar': 'South-eastern Asia',
    'Namibia': 'Southern Africa',
    'Nauru': 'Micronesia',
    'Nepal': 'Southern Asia',
    'Netherlands': 'Western Europe',
    'Nicaragua': 'Central America',
    'Niger': 'Western Africa',
    'Nigeria': 'Western Africa',
    'Norway': 'Northern Europe',
    'Oman': 'Western Asia',
    'Pakistan': 'Southern Asia',
    'Palau': 'Micronesia',
    'Panama': 'Central America',
    'Papua New Guinea': 'Melanesia',
    'Paraguay': 'South America',
    'People\'s Republic of Korea': 'Eastern Asia',
    'Peru': 'South America',
    'Philippines': 'South-eastern Asia',
    'Poland': 'Eastern Europe',
    'Portugal': 'Southern Europe',
    'Qatar': 'Western Asia',
    'Republic of Korea': 'Eastern Asia',
    'Romania': 'Eastern Europe',
    'Russia': 'Eastern Europe',
    'Rwanda': 'Eastern Africa',
    'Samoa': 'Polynesia',
    'Sao Tome and Principe': 'Middle Africa',
    'Saudi Arabia': 'Western Asia',
    'Senegal': 'Western Africa',
    'Serbia': 'Southern Europe',
    'Seychelles': 'Eastern Africa',
    'Sierra Leone': 'Western Africa',
    'Singapore': 'South-eastern Asia',
    'Slovakia': 'Eastern Europe',
    'Slovenia': 'Southern Europe',
    'Socialist Republic of Vietnam': 'South-eastern Asia',
    'Solomon Islands': 'Melanesia',
    'Somalia': 'Eastern Africa',
    'South Africa': 'Southern Africa',
    'South Sudan': 'Eastern Africa',
    'Spain': 'Southern Europe',
    'Sri Lanka': 'Southern Asia',
    'St. Kitts and Nevis': 'Caribbean',
    'St. Lucia': 'Caribbean',
    'St. Vincent and the Grenadines': 'Caribbean',
    'Sudan': 'Northern Africa',
    'Surinam': 'South America',
    'Swaziland': 'Southern Africa',
    'Sweden': 'Northern Europe',
    'Switzerland': 'Western Europe',
    'Syria': 'Western Asia',
    'Taiwan': 'Eastern Asia',
    'Tajikistan': 'Central Asia',
    'Tanzania': 'Eastern Africa',
    'Thailand': 'South-eastern Asia',
    'Togo': 'Western Africa',
    'Tonga': 'Polynesia',
    'Trinidad and Tobago': 'Caribbean',
    'Tunisia': 'Northern Africa',
    'Turkey': 'Western Asia',
    'Turkmenistan': 'Central Asia',
    'Tuvalu': 'Polynesia',
    'Uganda': 'Eastern Africa',
    'Ukraine': 'Eastern Europe',
    'United Arab Emirates': 'Western Asia',
    'United States of America': 'Northern America',
    'Uruguay': 'South America',
    'Uzbekistan': 'Central Asia',
    'Vanuatu': 'Melanesia',
    'Venezuela': 'South America',
    'Yemen': 'Western Asia',
    'Zambia': 'Eastern Africa',
    'Zimbabwe': 'Eastern Africa',
}


In [13]:
# Harmonize country_id names between the constitution files and V-Dem.
VDEM_COUNTRY_MAP = {
    'Bosnia-Herzegovina': 'Bosnia and Herzegovina',
    'Congo': 'Republic of the Congo',
    "Cote D'Ivoire": 'Ivory Coast',
    'Czech Republic': 'Czechia',
    'East Timor': 'Timor-Leste',
    'Gambia': 'The Gambia',
    'German Federal Republic': 'Germany',
    'Kyrgyz Republic': 'Kyrgyzstan',
    'Macedonia': 'North Macedonia',
    'Myanmar': 'Burma/Myanmar',
    "People's Republic of Korea": 'North Korea',
    'Republic of Korea': 'South Korea',
    'Socialist Republic of Vietnam': 'Vietnam',
    'Surinam': 'Suriname',
    'Swaziland': 'Eswatini',
    'Turkey': 'T?rkiye',
}


## Extract Metadata from Raw Files

Each constitution file contributes one document row. This function records the source filename, a document id, the country name stored in `country_id`, a UN geoscheme-style region/subregion label, the original creation year from the first year in the header, the amended/current version year from the filename, the file format, and basic size measures such as character and line count.

This distinction between `year_created` and `year_amended` is important because many constitutions in the corpus were originally adopted in one year but appear in an amended or revised version in the dataset.


In [14]:
def extract_doc_metadata(path: Path) -> dict:
    """Read one constitution file and return document-level metadata."""
    # Prefer header values when they are available, then fall back to the filename.
    text = path.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines()

    country_id = path.stem
    year_created = ""
    year_amended = ""

    # Use the filename as the first fallback source for country_id and amendment year.
    stem_match = RE_FILENAME_YEAR.match(path.stem)
    if stem_match:
        country_id = stem_match.group("country_id")
        year_amended = stem_match.group("year")

    if lines:
        match = RE_DOC_HEADER.match(lines[0].strip())
        if match:
            # Override fallback metadata with the more explicit document header.
            country_id = match.group(1).strip()
            year_created = match.group(2)
            extra_text = match.group(3) or ""
            extra_years = RE_ANY_YEAR.findall(extra_text)
            if not year_amended:
                # Use the latest extra year when an amended year appears in parentheses.
                year_amended = extra_years[-1] if extra_years else year_created

    return {
        "doc_id": path.stem,
        "country_id": country_id,
        "year_created": year_created,
        "year_amended": year_amended,
        "region": REGION_MAP.get(country_id, "Unknown"),
        "source_file": path.name,
        "file_format": "plaintext",
        "char_count": len(text),
        "line_count": len(lines),
    }


## Add V-Dem Features

This section loads selected regime indicators from the local `V-Dem-CY-Core-v16.csv` file and prepares categorical versions that are easier to use in later plots. The merge keys are `country_id` plus `year_amended`, because the constitution corpus reflects the amended version present in the dataset rather than the original enactment year.

The variables added are:
- `v2x_regime`
- `v2x_freexp_altinf`
- `v2x_rule`
- `v2x_regime_cat`
- `v2x_freexp_altinf_cat`
- `v2x_rule_cat`

A small country-name harmonization map for the `country_id` field handles cases where the constitution filenames and V-Dem use different labels. If a country_id-year is missing in the local V-Dem extract, the added fields remain blank in `lib.csv`.


In [15]:
# Load the V-Dem indicators needed for the LIB merge.
def load_vdem_features(vdem_csv: Path) -> pd.DataFrame:
    """Load selected V-Dem variables and harmonize country names for the `country_id` merge field."""
    if not vdem_csv.exists():
        return pd.DataFrame()

    # Read only the columns needed for the project-level join.
    vdem = pd.read_csv(
        vdem_csv,
        usecols=['country_name', 'country_text_id', 'year', 'v2x_regime', 'v2x_freexp_altinf', 'v2x_rule'],
        encoding='utf-8',
    )
    vdem = vdem.rename(columns={
        'country_name': 'vdem_country_name',
        'year': 'year_amended',
    })
    vdem['year_amended'] = vdem['year_amended'].astype(str)
    # Start from the V-Dem country label for the `country_id` field, then overwrite mismatches via the map.
    vdem['country_id'] = vdem['vdem_country_name']
    for lib_name, vdem_name in VDEM_COUNTRY_MAP.items():
        vdem.loc[vdem['vdem_country_name'] == vdem_name, 'country_id'] = lib_name
    # Normalize Turkey separately because V-Dem uses a text id that changed labels.
    vdem.loc[vdem['country_text_id'] == 'TUR', 'country_id'] = 'Turkey'
    return vdem[['country_id', 'year_amended', 'v2x_regime', 'v2x_freexp_altinf', 'v2x_rule']]


In [16]:
# Convert V-Dem scores into graph-friendly categorical labels.
V2X_REGIME_CATS = {
    0: 'Closed autocracy',
    1: 'Electoral autocracy',
    2: 'Electoral democracy',
    3: 'Liberal democracy',
}

V2X_SCORE_LABELS = ['Very low', 'Low', 'Moderate', 'High', 'Very high']
V2X_SCORE_BINS = [-0.000001, 0.2, 0.4, 0.6, 0.8, 1.000001]

REGION_COMPRESSED_MAP = {
    'Eastern Africa': 'Sub-Saharan Africa',
    'Middle Africa': 'Sub-Saharan Africa',
    'Northern Africa': 'North Africa',
    'Southern Africa': 'Sub-Saharan Africa',
    'Western Africa': 'Sub-Saharan Africa',
    'Caribbean': 'Caribbean',
    'Central America': 'North America',
    'Northern America': 'North America',
    'South America': 'South America',
    'Eastern Asia': 'Asia',
    'Southern Asia': 'Asia',
    'South-eastern Asia': 'Asia',
    'Eastern Europe': 'Eastern Europe',
    'Northern Europe': 'Western Europe',
    'Southern Europe': 'Western Europe',
    'Western Europe': 'Western Europe',
    'Central Asia': 'Middle East/Central Asia',
    'Western Asia': 'Middle East/Central Asia',
    'Australia and New Zealand': 'Oceania',
    'Melanesia': 'Oceania',
    'Micronesia': 'Oceania',
    'Polynesia': 'Oceania',
}

def add_vdem_categories(lib: pd.DataFrame) -> pd.DataFrame:
    """Add categorical V-Dem fields for plotting and grouped summaries."""
    lib = lib.copy()
    lib['region_compressed'] = lib['region'].map(REGION_COMPRESSED_MAP).fillna(lib['region'])
    lib['v2x_regime_cat'] = lib['v2x_regime'].round().map(V2X_REGIME_CATS).fillna('Unknown')
    lib['v2x_freexp_altinf_cat'] = pd.cut(
        lib['v2x_freexp_altinf'],
        bins=V2X_SCORE_BINS,
        labels=V2X_SCORE_LABELS,
    ).astype('object').fillna('Unknown')
    lib['v2x_rule_cat'] = pd.cut(
        lib['v2x_rule'],
        bins=V2X_SCORE_BINS,
        labels=V2X_SCORE_LABELS,
    ).astype('object').fillna('Unknown')
    return lib


## Add Parser-Derived Features

If `corpus.csv` is available, this section aggregates the parsed token-level corpus back to one row per constitution so `LIB` also includes parser-derived size measures:

- `article_count`: the maximum normalized `article_n` value observed in the document, which counts article-like legal units rather than only literal source headings named `Article`
- `clause_count`: the number of unique article/clause combinations in the document
- `token_count`: the maximum token number observed in the document

These fields make the final `LIB` table more useful for document-level comparison, filtering, and later visualization work.


In [17]:
# Summarize parser output into one row per constitution.
def load_parser_features(corpus_csv: Path) -> pd.DataFrame:
    """Aggregate article, clause, and token counts from the parsed corpus."""
    if not corpus_csv.exists():
        return pd.DataFrame()

    corpus = pd.read_csv(corpus_csv, usecols=['country_id', 'article_n', 'clause_n', 'token_n'])

    article_counts = (
        corpus.groupby('country_id')['article_n']
        .max()
        .rename('article_count')
    )
    clause_counts = (
        corpus[['country_id', 'article_n', 'clause_n']]
        .drop_duplicates()
        .groupby('country_id')
        .size()
        .rename('clause_count')
    )
    token_counts = (
        corpus.groupby('country_id')['token_n']
        .max()
        .rename('token_count')
    )

    # Merge the parser-derived size measures into one document-level table.
    parser_features = pd.concat([article_counts, clause_counts, token_counts], axis=1).reset_index()
    return parser_features


## Build the LIB Table

This function combines raw-file metadata with parser-derived features and V-Dem regime indicators, then sorts the result into a stable document-level table. The final output is a single `LIB` DataFrame in which each row represents one constitution and each column stores either document metadata, document-level summary counts, or matched country-year regime measures keyed by `country_id`.


In [18]:
# Combine raw-file metadata, parser features, and V-Dem indicators.
def build_lib(folder: str, corpus_csv: str = "corpus.csv", vdem_csv: str = "V-Dem-CY-Core-v16.csv") -> pd.DataFrame:
    """Create the LIB table from raw constitution files."""
    # Read the raw constitution files in a stable order.
    files = sorted(Path(folder).glob("*.txt"))
    rows = [extract_doc_metadata(path) for path in files]
    lib = pd.DataFrame(rows)
    # Cast year columns to strings before merging with CSV-based metadata.
    lib["year_created"] = lib["year_created"].astype(str)
    lib["year_amended"] = lib["year_amended"].astype(str)

    parser_features = load_parser_features(Path(corpus_csv))
    if not parser_features.empty:
        # Add article, clause, and token counts from the parsed corpus.
        lib = lib.merge(parser_features, on=["country_id"], how="left")

    vdem_features = load_vdem_features(Path(vdem_csv))
    if not vdem_features.empty:
        # Add V-Dem indicators by constitution country_id and amended year.
        lib = lib.merge(vdem_features, on=["country_id", "year_amended"], how="left")
        lib = add_vdem_categories(lib)

    # Keep the compressed region next to the original region in the final export.
    if 'region_compressed' in lib.columns:
        cols = lib.columns.tolist()
        cols.insert(cols.index('region') + 1, cols.pop(cols.index('region_compressed')))
        lib = lib[cols]

    # Keep the final table sorted for easier inspection and reproducible output.
    lib = lib.sort_values(["country_id", "year_amended"]).reset_index(drop=True)
    return lib


## Create and Preview the Table

Run this cell to build the `LIB` DataFrame and preview a few rows. This is a good place to confirm that countries, regions, creation years, amended years, and size measures all look correct before exporting the CSV.


In [19]:
# Preview the assembled LIB table before saving it.
lib = build_lib('individual_constitutions', 'corpus.csv', 'V-Dem-CY-Core-v16.csv')
print(f'LIB rows: {len(lib)}')
print(f'Average document length (characters): {lib["char_count"].mean():.2f}')
lib.head(10)


LIB rows: 192
Average document length (characters): 138099.46


,doc_id,country_id,year_created,year_amended,region,region_compressed,source_file,file_format,char_count,line_count,article_count,clause_count,token_count,v2x_regime,v2x_freexp_altinf,v2x_rule,v2x_regime_cat,v2x_freexp_altinf_cat,v2x_rule_cat
0,Afghanistan_2004,Afghanistan,2004,2004,Southern Asia,Asia,Afghanistan_2004.txt,plaintext,66806,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
1,Albania_2008,Albania,1998,2008,Southern Europe,Western Europe,Albania_2008.txt,plaintext,86022,3327,183,582,13221,2.0,0.807,0.516,Electoral democracy,Very high,Moderate
2,Algeria_2008,Algeria,1963,2008,Northern Africa,North Africa,Algeria_2008.txt,plaintext,66590,2098,185,460,10072,1.0,0.662,0.250,Electoral autocracy,High,Low
3,Andorra_1993,Andorra,1993,1993,Southern Europe,Western Europe,Andorra_1993.txt,plaintext,55687,1668,108,277,8310,NaN,NaN,NaN,Unknown,Unknown,Unknown
4,Angola_2010,Angola,2010,2010,Middle Africa,Sub-Saharan Africa,Angola_2010.txt,plaintext,175148,5274,250,1123,24586,1.0,0.361,0.129,Electoral autocracy,Low,Very low
5,Antigua_and_Barbuda_1981,Antigua and Barbuda,1981,1981,Caribbean,Caribbean,Antigua_and_Barbuda_1981.txt,plaintext,214275,4229,130,963,35033,NaN,NaN,NaN,Unknown,Unknown,Unknown
6,Argentina_1994,Argentina,1853,1994,South America,South America,Argentina_1994.txt,plaintext,82255,1455,131,301,12571,2.0,0.944,0.523,Electoral democracy,Very high,Moderate
7,Armenia_2005,Armenia,1995,2005,Western Asia,Middle East/Central Asia,Armenia_2005.txt,plaintext,86487,1961,144,484,13312,1.0,0.584,0.221,Electoral autocracy,Moderate,Low
8,Australia_1985,Australia,1901,1985,Australia and New Zealand,Oceania,Australia_1985.txt,plaintext,81224,1661,145,359,12384,3.0,0.962,0.993,Liberal democracy,Very high,Very high
9,Austria_2009,Austria,1920,2009,Western Europe,Western Europe,Austria_2009.txt,plaintext,275162,5091,227,954,43293,3.0,0.966,0.967,Liberal democracy,Very high,Very high


## Save the LIB Table

This final cell writes the document-level table to `lib.csv` for use in the rest of the final project. Once saved, this file becomes the main source for document metadata used in later analyses and visualizations.


In [20]:
# Save the final LIB table for downstream notebooks.
output_csv = Path('lib.csv')
lib.to_csv(output_csv, index=False)
print(f'Saved {len(lib)} LIB rows to: {output_csv.resolve()}')
print('Columns:', ', '.join(lib.columns))


Saved 192 LIB rows to: C:\Users\garre\school\spring_2026\ds_5001\lib.csv
Columns: doc_id, country_id, year_created, year_amended, region, region_compressed, source_file, file_format, char_count, line_count, article_count, clause_count, token_count, v2x_regime, v2x_freexp_altinf, v2x_rule, v2x_regime_cat, v2x_freexp_altinf_cat, v2x_rule_cat
